# Sphere Function Optimization

## 1. CUDA Parallel Implementation

In [ ]:
%%writefile sphere_cuda.cu
#include <iostream>
#include <cmath>
#include <fstream>
#include <curand_kernel.h>
#include <cfloat>
#include <chrono>

using namespace std;

#define POP 256
#define DIM 5
#define MAX_IT 8000
#define LB -100.0
#define UB 100.0

//======================== RANDOM ========================//
__device__ double randF(curandState* state,double a,double b){
    double r=curand_uniform_double(state);
    return a+r*(b-a);
}
__device__ int randInt(curandState* state,int a,int b){
    float r=curand_uniform(state);
    return (int)(a+r*(b-a+0.99999));
}

//======================== SPHERE FITNESS ========================//
__device__ double fitness(double* x,int dim){
    double sum=0;
    for(int i=0;i<dim;i++) sum+=x[i]*x[i];
    return sum;
}

//======================== INIT POP ========================//
__global__ void init_population_kernel(double* pop,curandState* states,unsigned long seed){
    int idx=blockIdx.x*blockDim.x+threadIdx.x;
    if(idx<POP){
        curand_init(seed,idx,0,&states[idx]);
        for(int d=0;d<DIM;d++) pop[idx*DIM+d]=randF(&states[idx],LB,UB);
    }
}

//======================== EVAL ========================//
__global__ void evaluate_fitness_kernel(double* pop,double* fitness_vals){
    int idx=blockIdx.x*blockDim.x+threadIdx.x;
    if(idx<POP){
        double temp[DIM];
        for(int d=0;d<DIM;d++) temp[d]=pop[idx*DIM+d];
        fitness_vals[idx]=fitness(temp,DIM);
    }
}

//======================== LOA UPDATE ========================//
__global__ void loa_update_kernel(double* pop,double* fitness_vals,double* new_pop,double* new_fitness_vals,curandState* states,int t){
    int idx=blockIdx.x*blockDim.x+threadIdx.x;
    if(idx<POP){
        curandState localState=states[idx];

        int betterCount=0,betterIdx=-1;
        for(int j=0;j<POP;j++) if(fitness_vals[j]<fitness_vals[idx]) betterCount++;

        if(betterCount>0){
            int pick=randInt(&localState,0,betterCount-1),c=0;
            for(int j=0;j<POP;j++){
                if(fitness_vals[j]<fitness_vals[idx]&&c++==pick){betterIdx=j;break;}
            }
        }

        double curr[DIM],cand[DIM];
        for(int d=0;d<DIM;d++) curr[d]=pop[idx*DIM+d];

        double rp=curand_uniform_double(&localState);
        if(rp<0.5&&betterIdx!=-1){
            for(int d=0;d<DIM;d++){
                double r=curand_uniform_double(&localState);
                int I=randInt(&localState,1,2);
                cand[d]=curr[d]+r*(pop[betterIdx*DIM+d]-I*curr[d]);
                cand[d]=max(LB,min(UB,cand[d]));
            }
        }else{
            for(int d=0;d<DIM;d++){
                double r=curand_uniform_double(&localState);
                cand[d]=curr[d]+(1-2*r)*(UB-LB)/double(t);
                cand[d]=max(LB,min(UB,cand[d]));
            }
        }

        double f=fitness(cand,DIM);
        if(f<fitness_vals[idx]){
            for(int d=0;d<DIM;d++) new_pop[idx*DIM+d]=cand[d];
            new_fitness_vals[idx]=f;
        }else{
            for(int d=0;d<DIM;d++) new_pop[idx*DIM+d]=curr[d];
            new_fitness_vals[idx]=fitness_vals[idx];
        }
        states[idx]=localState;
    }
}

//======================== MAIN ========================//
int main(){
    auto start=chrono::high_resolution_clock::now();

    ofstream csv("sphere_log.csv");
    csv<<"iter,best";
    for(int d=1;d<=DIM;d++) csv<<",x"<<d;
    csv<<"\n";

    double *d_pop,*d_fit,*d_new_pop,*d_new_fit;
    curandState *d_states;

    size_t PS=POP*DIM*sizeof(double);
    size_t FS=POP*sizeof(double);

    cudaMalloc(&d_pop,PS);
    cudaMalloc(&d_fit,FS);
    cudaMalloc(&d_new_pop,PS);
    cudaMalloc(&d_new_fit,FS);
    cudaMalloc(&d_states,POP*sizeof(curandState));

    int threads=128,blocks=(POP+threads-1)/threads;

    init_population_kernel<<<blocks,threads>>>(d_pop,d_states,time(NULL));
    evaluate_fitness_kernel<<<blocks,threads>>>(d_pop,d_fit);
    cudaDeviceSynchronize();

    double bestGlobalFit=DBL_MAX;
    double h_pop[POP*DIM],h_fit[POP];

    for(int it=1;it<=MAX_IT;it++){

        loa_update_kernel<<<blocks,threads>>>(d_pop,d_fit,d_new_pop,d_new_fit,d_states,it);
        cudaDeviceSynchronize();

        swap(d_pop,d_new_pop);
        swap(d_fit,d_new_fit);

        cudaMemcpy(h_fit,d_fit,FS,cudaMemcpyDeviceToHost);
        cudaMemcpy(h_pop,d_pop,PS,cudaMemcpyDeviceToHost);

        int bestIdx=0;
        for(int i=1;i<POP;i++) if(h_fit[i]<h_fit[bestIdx]) bestIdx=i;

        bestGlobalFit=min(bestGlobalFit,h_fit[bestIdx]);

        // Write EVERY ITERATION to CSV
        csv<<it<<","<<bestGlobalFit;
        for(int d=0;d<DIM;d++) csv<<","<<h_pop[bestIdx*DIM+d];
        csv<<"\n";

        // Keep your original printed output behavior ✔
        if(it%1000==0) printf("Iter %d | Best = %f\n",it,bestGlobalFit);
    }

    csv.close();

    // Final results (unchanged)
    int bi=0; for(int i=1;i<POP;i++) if(h_fit[i]<h_fit[bi]) bi=i;

    cout<<"\nFinal Best Solution:\n";
    for(int d=0;d<DIM;d++) cout<<"x"<<d+1<<" = "<<h_pop[bi*DIM+d]<<endl;
    cout<<"\nBest Sphere Value = "<<bestGlobalFit<<endl;

    auto end=chrono::high_resolution_clock::now();
    cout<<"\nExecution Time = "<<chrono::duration<double>(end-start).count()<<" sec\n";

    return 0;
}


Writing sphere_cuda.cu


In [ ]:
!nvcc -arch=sm_75 sphere_cuda.cu -o sphere_cuda
!./sphere_cuda

Iter 1000 | Best = 0.000000
Iter 2000 | Best = 0.000000
Iter 3000 | Best = 0.000000
Iter 4000 | Best = 0.000000
Iter 5000 | Best = 0.000000
Iter 6000 | Best = 0.000000
Iter 7000 | Best = 0.000000
Iter 8000 | Best = 0.000000

Final Best Solution:
x1 = 4.63443e-163
x2 = -7.0089e-163
x3 = -3.25676e-163
x4 = -3.7063e-163
x5 = -6.88513e-163

Best Sphere Value = 0

Execution Time = 0.747855 sec


## 2. Serial (Sequential) Implementation

In [ ]:

%%writefile sphere_serial.cpp
#include <bits/stdc++.h>
using namespace std;

/* ---------------------------------------
      Random Generator
---------------------------------------*/
mt19937 rng(time(NULL));

double randF(double a, double b) {
    uniform_real_distribution<double> dist(a, b);
    return dist(rng);
}
int randInt(int a, int b) {
    uniform_int_distribution<int> dist(a, b);
    return dist(rng);
}

/* ---------------------------------------
            Fitness Function
---------------------------------------*/
double sphere(const vector<double> &x) {
    int dim = x.size();

    double sum = 0.0;
    for (int i = 0; i < dim; ++i) {
        sum += x[i] * x[i];
    }
    return sum;

}

/* ---------------------------------------
            LOA PARAMETERS
---------------------------------------*/
int POP = 256;
int DIM = 5;
int MAX_IT = 8000;
double LB = -100;
double UB = 100;

/* ---------------------------------------
     Escape (Global Search)
---------------------------------------*/
vector<double> escape(const vector<double> &x,
                      const vector<double> &SSA)
{
    vector<double> newX = x;
    for(int j=0;j<DIM;j++){
        double r = randF(LB, UB); // Note: Original code used randF(rng,0,1) here but logic was r*(SSA-I*x).
        // Wait, original code: double r = randF(rng,0,1);
        // Let's stick to original logic.
        double r_val = randF(0, 1);
        int I = randInt(1, 2);
        newX[j] = x[j] + r_val * (SSA[j] - I*x[j]);

        newX[j] = min(max(newX[j],LB),UB);
    }
    return newX;
}

/* ---------------------------------------
     Hide (Local Search)
---------------------------------------*/
vector<double> hide(const vector<double> &Xi,int t){
    vector<double> newX = Xi;

    for(int j=0;j<DIM;j++){
        double r = randF(0, 1);
        newX[j] = Xi[j] + (1 - 2*r)*(UB-LB)/t;
        newX[j] = min(max(newX[j],LB),UB);
    }
    return newX;
}

/* ---------------------------------------
         Initialize population
---------------------------------------*/
vector<vector<double>> init_population(){
    vector<vector<double>> pop(POP, vector<double>(DIM));
    for(int i=0;i<POP;i++)
        for(int d=0;d<DIM;d++)
            pop[i][d] = randF(LB, UB);
    return pop;
}

/* =======================================
             MAIN LOA SERIAL
=======================================*/
int main(){
    auto t1 = chrono::high_resolution_clock::now();

    vector<vector<double>> pop = init_population();
    vector<double> fitness(POP);

    // Initial fitness
    for(int i=0;i<POP;i++) fitness[i]=sphere(pop[i]);

    double bestFit = 1e18;
    vector<double> bestSol(DIM);

    // find initial best
    for(int i=0;i<POP;i++){
        if(fitness[i]<bestFit){
            bestFit=fitness[i];
            bestSol=pop[i];
        }
    }

    /* --------------------------------------
                LOA ITERATIONS
    ---------------------------------------*/
    for(int it=1; it<=MAX_IT; it++){

        for(int i=0;i<POP;i++){

            vector<int> better;
            for(int j=0;j<POP;j++)
                if(fitness[j]<fitness[i]) better.push_back(j);

            int betterIdx=-1;
            if(!better.empty())
                betterIdx = better[randInt(0,(int)better.size()-1)];

            vector<double> candidate;
            if(randF(0,1)<0.5 && betterIdx!=-1)
                candidate = escape(pop[i],pop[betterIdx]);
            else
                candidate = hide(pop[i],it);

            double f = sphere(candidate);

            if(f<fitness[i]){
                pop[i]=candidate;
                fitness[i]=f;
            }
            if(f<bestFit){
                bestFit=f;
                bestSol=candidate;
            }
        }

        if(it % 1000 == 0)
            cout<<"Iter "<<it<<" | Best = "<<bestFit<<"\n";
    }

    cout<<"\nFinal Best Solution:\n";
    for(int i=0;i<DIM;i++) cout<<"x"<<i+1<<" = "<<bestSol[i]<<endl;
    cout<<"\nBest " << "Sphere" << " Value = "<<bestFit<<endl;

    auto t2 = chrono::high_resolution_clock::now();
    cout<<"\nExecution Time = "
        <<chrono::duration<double>(t2-t1).count()
        <<" sec\n";

    return 0;
}


Writing sphere_serial.cpp


In [ ]:
!g++ sphere_serial.cpp -o sphere_serial
!./sphere_serial

Iter 1000 | Best = 4.70403e-271
Iter 2000 | Best = 0
Iter 3000 | Best = 0
Iter 4000 | Best = 0
Iter 5000 | Best = 0
Iter 6000 | Best = 0
Iter 7000 | Best = 0
Iter 8000 | Best = 0

Final Best Solution:
x1 = 1.46574e-162
x2 = 2.84236e-163
x3 = 8.19984e-163
x4 = 3.4595e-163
x5 = -1.26495e-162

Best Sphere Value = 0

Execution Time = 7.34009 sec


## 3. OpenMP Parallel Implementation

In [ ]:

%%writefile sphere_omp.cpp
#include <bits/stdc++.h>
#include <omp.h>
using namespace std;

/* ---------------------------------------
      Thread-safe Random Generator
---------------------------------------*/
double randF(mt19937 &rng, double a, double b) {
    uniform_real_distribution<double> dist(a, b);
    return dist(rng);
}
int randInt(mt19937 &rng, int a, int b) {
    uniform_int_distribution<int> dist(a, b);
    return dist(rng);
}

/* ---------------------------------------
            Fitness Function
---------------------------------------*/
double sphere(const vector<double> &x) {
    int dim = x.size();

    double sum = 0.0;
    for (int i = 0; i < dim; ++i) {
        sum += x[i] * x[i];
    }
    return sum;

}

/* ---------------------------------------
            LOA PARAMETERS
---------------------------------------*/
int POP = 256;
int DIM = 5;
int MAX_IT = 8000;
double LB = -100;
double UB = 100;

/* ---------------------------------------
     Escape (Global Search)  — parallel safe
---------------------------------------*/
vector<double> escape(const vector<double> &x,
                      const vector<double> &SSA,
                      mt19937 &rng)
{
    vector<double> newX = x;
    for(int j=0;j<DIM;j++){
        double r = randF(rng,0,1);
        int I = randInt(rng,1,2);
        newX[j] = x[j] + r * (SSA[j] - I*x[j]);

        newX[j] = min(max(newX[j],LB),UB);
    }
    return newX;
}

/* ---------------------------------------
     Hide (Local Search) — parallel safe
---------------------------------------*/
vector<double> hide(const vector<double> &Xi,int t,mt19937 &rng){
    vector<double> newX = Xi;

    for(int j=0;j<DIM;j++){
        double r = randF(rng,0,1);
        newX[j] = Xi[j] + (1 - 2*r)*(UB-LB)/t;
        newX[j] = min(max(newX[j],LB),UB);
    }
    return newX;
}

/* ---------------------------------------
         Initialize population (PARALLEL)
---------------------------------------*/
vector<vector<double>> init_population(vector<mt19937> &rngs){
    vector<vector<double>> pop(POP, vector<double>(DIM));

    #pragma omp parallel
    {
        int tid = omp_get_thread_num();
        mt19937 &local_rng = rngs[tid];

        #pragma omp for schedule(static)
        for(int i=0;i<POP;i++)
            for(int d=0;d<DIM;d++)
                pop[i][d] = randF(local_rng,LB,UB);
    }
    return pop;
}

/* =======================================
             MAIN LOA PARALLEL
=======================================*/
int main(){
    auto t1 = chrono::high_resolution_clock::now();

    int threads = omp_get_max_threads();
    vector<mt19937> rngs(threads);

    random_device rd;
    for(int i=0;i<threads;i++)
        rngs[i].seed(rd()+i*111);

    vector<vector<double>> pop = init_population(rngs);
    vector<double> fitness(POP);

    // Initial fitness
    for(int i=0;i<POP;i++) fitness[i]=sphere(pop[i]);

    double bestFit = 1e18;
    vector<double> bestSol(DIM);

    // find initial best
    for(int i=0;i<POP;i++){
        if(fitness[i]<bestFit){
            bestFit=fitness[i];
            bestSol=pop[i];
        }
    }

    /* --------------------------------------
                LOA ITERATIONS
       Full population parallel every step
    ---------------------------------------*/
    for(int it=1; it<=MAX_IT; it++){

        #pragma omp parallel
        {
            int tid = omp_get_thread_num();
            mt19937 &localRng = rngs[tid];

            double localBest = 1e18;
            vector<double> localBestSol(DIM);

            #pragma omp for schedule(static)
            for(int i=0;i<POP;i++){

                vector<int> better;
                for(int j=0;j<POP;j++)
                    if(fitness[j]<fitness[i]) better.push_back(j);

                int betterIdx=-1;
                if(!better.empty())
                    betterIdx = better[randInt(localRng,0,(int)better.size()-1)];

                vector<double> candidate;
                if(randF(localRng,0,1)<0.5 && betterIdx!=-1)
                    candidate = escape(pop[i],pop[betterIdx],localRng);
                else
                    candidate = hide(pop[i],it,localRng);

                double f = sphere(candidate);

                if(f<fitness[i]){
                    pop[i]=candidate;
                    fitness[i]=f;
                }
                if(f<localBest){
                    localBest=f;
                    localBestSol=candidate;
                }
            }

            // Update global best safely
            #pragma omp critical
            {
                if(localBest<bestFit){
                    bestFit=localBest;
                    bestSol=localBestSol;
                }
            }
        }

        if(it % 1000 == 0)
            cout<<"Iter "<<it<<" | Best = "<<bestFit<<"\n";
    }

    cout<<"\nFinal Best Solution:\n";
    for(int i=0;i<DIM;i++) cout<<"x"<<i+1<<" = "<<bestSol[i]<<endl;
    cout<<"\nBest " << "Sphere" << " Value = "<<bestFit<<endl;

    auto t2 = chrono::high_resolution_clock::now();
    cout<<"\nExecution Time = "
        <<chrono::duration<double>(t2-t1).count()
        <<" sec\n";

    return 0;
}


Writing sphere_omp.cpp


In [ ]:
!g++ -fopenmp sphere_omp.cpp -o sphere_omp
!./sphere_omp

Iter 1000 | Best = 4.31777e-270
Iter 2000 | Best = 0
Iter 3000 | Best = 0
Iter 4000 | Best = 0
Iter 5000 | Best = 0
Iter 6000 | Best = 0
Iter 7000 | Best = 0
Iter 8000 | Best = 0

Final Best Solution:
x1 = 1.03949e-163
x2 = -8.49531e-163
x3 = -1.28862e-163
x4 = 5.37147e-163
x5 = -8.62719e-163

Best Sphere Value = 0

Execution Time = 6.48816 sec
